In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell F3A.1 - Overview and exact paths
# Purpose:
# Annotate the three individual sequence patterns that survived
# Benjamini-Hochberg correction in Notebook 17.
#
# Fixed significant patterns:
# 7319, 498705, 2846
#
# This notebook does NOT perform another association analysis.
# It resolves:
# - the unitig(s) belonging to each significant pattern;
# - their sequences;
# - their existing Notebook 19 reference-mapping information;
# - carrier count and MIC-association direction.
#
# After the exact reference positions are known, nearby/overlapping genes
# can be annotated without guessing.

from pathlib import Path
import gzip
import json

import numpy as np
import pandas as pd
from IPython.display import display
PROJECT_ROOT = _repo_root()

RESULTS_TABLE_DIR = (
    PROJECT_ROOT
    / "05_Results"
    / "Tables"
)

NB17_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "17_Full_Chromosomal_Unitig_Pattern_Association"
)

NB11_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "11_Unitig_Patterns"
)

NB10_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "10_Whole_Chromosome_Unitigs"
)

NB19_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "19_Broad_Unitig_Ablation"
)

NB17_RESULTS = (
    NB17_DIR
    / "17_all_unique_pattern_association_results.csv.gz"
)

UNITIG_TO_PATTERN_FILE = (
    NB11_DIR
    / "11_unitig_to_pattern.npz"
)

UNITIG_FASTA = (
    NB10_DIR
    / "10_variable_unitigs.fasta.gz"
)

NB19_ASSIGNMENT = (
    NB19_DIR
    / "19_unitig_reference_assignment.npz"
)

OUTPUT_TABLE = (
    RESULTS_TABLE_DIR
    / "Figure03_three_BH_significant_pattern_annotation.csv"
)

QC_FILE = (
    RESULTS_TABLE_DIR
    / "Figure03_three_BH_significant_pattern_annotation_QC.csv"
)

SIGNIFICANT_PATTERN_IDS = np.array(
    [7319, 498705, 2846],
    dtype=int,
)

EXPECTED_UNITIGS = 1_287_844

for path in [
    NB17_RESULTS,
    UNITIG_TO_PATTERN_FILE,
    UNITIG_FASTA,
    NB19_ASSIGNMENT,
]:
    assert path.exists(), f"Required input not found: {path}"

print("Figure 3 significant-pattern annotation")
print("Patterns:", SIGNIFICANT_PATTERN_IDS.tolist())

print(
    "\nTransition: Cell F3A.2 will verify the three patterns directly "
    "from the authoritative Notebook 17 result table."
)


In [ ]:
#@title Cell F3A.2 - Verify the three BH-significant patterns
# Purpose:
# Confirm the three significant patterns and their saved association statistics.

results17 = pd.read_csv(
    NB17_RESULTS
)

assert len(results17) == 504_889

required_columns = [
    "pattern_id",
    "first_unitig_index",
    "n_unitigs",
    "present_count",
    "high_MIC_count",
    "remaining_count",
    "estimated_log2_MIC_difference",
    "p_value_two_sided",
    "Benjamini_Hochberg_q_value",
    "association_after_Benjamini_Hochberg_0_05",
    "estimated_direction",
]

for column in required_columns:
    assert column in results17.columns, (
        f"Missing Notebook 17 column: {column}"
    )

significant = (
    results17.loc[
        results17[
            "association_after_Benjamini_Hochberg_0_05"
        ].astype(bool)
    ]
    .copy()
    .sort_values(
        "Benjamini_Hochberg_q_value"
    )
    .reset_index(drop=True)
)

assert len(significant) == 3

assert set(
    significant[
        "pattern_id"
    ].astype(int)
) == set(
    SIGNIFICANT_PATTERN_IDS.tolist()
)

print("Notebook 17 verification: PASS")

display(
    significant[
        [
            "pattern_id",
            "first_unitig_index",
            "n_unitigs",
            "present_count",
            "high_MIC_count",
            "remaining_count",
            "estimated_log2_MIC_difference",
            "p_value_two_sided",
            "Benjamini_Hochberg_q_value",
            "estimated_direction",
        ]
    ]
)

print(
    "\nTransition: Cell F3A.3 will resolve the exact unitig indices "
    "belonging to each significant carrier pattern."
)


In [ ]:
#@title Cell F3A.3 - Resolve unitigs belonging to each significant pattern
# Purpose:
# Identify the authoritative unitig-to-pattern array without guessing its NPZ key.
#
# A valid array must be:
# - one-dimensional;
# - length 1,287,844;
# - integer-like;
# - map each Notebook 17 first_unitig_index back to its recorded pattern_id.

with np.load(
    UNITIG_TO_PATTERN_FILE
) as archive:
    candidate_arrays = []

    for key in archive.files:
        arr = np.asarray(
            archive[key]
        )

        if (
            arr.ndim == 1
            and len(arr) == EXPECTED_UNITIGS
            and np.issubdtype(
                arr.dtype,
                np.integer,
            )
        ):
            candidate_arrays.append(
                (
                    key,
                    arr.astype(
                        np.int64,
                        copy=False,
                    ),
                )
            )

print(
    "Candidate 1D integer arrays of unitig length:",
    [x[0] for x in candidate_arrays],
)

valid_arrays = []

for key, arr in candidate_arrays:
    checks = []

    for _, row in significant.iterrows():
        first_index = int(
            row[
                "first_unitig_index"
            ]
        )

        pattern_id = int(
            row[
                "pattern_id"
            ]
        )

        checks.append(
            int(
                arr[
                    first_index
                ]
            )
            == pattern_id
        )

    if all(checks):
        valid_arrays.append(
            (
                key,
                arr,
            )
        )

assert len(valid_arrays) == 1, (
    "Could not uniquely identify the authoritative unitig-to-pattern array.\n"
    f"Valid keys: {[x[0] for x in valid_arrays]}"
)

UNITIG_TO_PATTERN_KEY, unitig_to_pattern = valid_arrays[0]

print(
    "Authoritative unitig-to-pattern key:",
    UNITIG_TO_PATTERN_KEY,
)

pattern_unitig_indices = {}

for _, row in significant.iterrows():
    pattern_id = int(
        row[
            "pattern_id"
        ]
    )

    indices = np.flatnonzero(
        unitig_to_pattern
        == pattern_id
    )

    expected_n = int(
        row[
            "n_unitigs"
        ]
    )

    assert len(indices) == expected_n, (
        f"Pattern {pattern_id}: expected {expected_n} unitigs, "
        f"found {len(indices)}."
    )

    assert int(
        row[
            "first_unitig_index"
        ]
    ) in set(
        indices.tolist()
    )

    pattern_unitig_indices[
        pattern_id
    ] = indices

    print(
        f"Pattern {pattern_id}: "
        f"{len(indices)} unitig(s) -> {indices.tolist()}"
    )

print(
    "\nTransition: Cell F3A.4 will retrieve the exact unitig sequences."
)


In [ ]:
#@title Cell F3A.4 - Retrieve exact unitig sequences
# Purpose:
# Stream the authoritative compressed FASTA once and retrieve only the
# unitigs belonging to the three significant patterns.

wanted_indices = set(
    np.concatenate(
        list(
            pattern_unitig_indices.values()
        )
    ).astype(
        int
    ).tolist()
)

sequence_by_index = {}

def fasta_records(handle):
    header = None
    sequence_parts = []

    for line in handle:
        line = line.strip()

        if not line:
            continue

        if line.startswith(">"):
            if header is not None:
                yield (
                    header,
                    "".join(
                        sequence_parts
                    ),
                )

            header = line[1:]
            sequence_parts = []

        else:
            sequence_parts.append(
                line
            )

    if header is not None:
        yield (
            header,
            "".join(
                sequence_parts
            ),
        )

with gzip.open(
    UNITIG_FASTA,
    "rt",
    encoding="utf-8",
) as handle:

    for unitig_index, (
        header,
        sequence,
    ) in enumerate(
        fasta_records(
            handle
        )
    ):
        if unitig_index in wanted_indices:
            sequence_by_index[
                unitig_index
            ] = {
                "header": header,
                "sequence": sequence,
            }

        if len(
            sequence_by_index
        ) == len(
            wanted_indices
        ):
            break

assert set(
    sequence_by_index
) == wanted_indices, (
    "Not all significant-pattern unitigs were recovered from the FASTA."
)

print(
    "Recovered significant-pattern unitig sequences:",
    len(
        sequence_by_index
    ),
)

for pattern_id in SIGNIFICANT_PATTERN_IDS:
    print(
        f"\nPattern {pattern_id}"
    )

    for unitig_index in pattern_unitig_indices[
        int(
            pattern_id
        )
    ]:
        record = sequence_by_index[
            int(
                unitig_index
            )
        ]

        print(
            " unitig_index =",
            int(
                unitig_index
            ),
            " length =",
            len(
                record[
                    "sequence"
                ]
            ),
            " header =",
            record[
                "header"
            ],
        )

print(
    "\nTransition: Cell F3A.5 will attach the existing Notebook 19 "
    "reference-mapping fields to these unitigs."
)


In [ ]:
#@title Cell F3A.5 - Attach Notebook 19 reference-mapping information
# Purpose:
# Read all unitig-length arrays already saved by Notebook 19 and extract
# their values only for the significant-pattern unitigs.
#
# This avoids guessing Notebook 19 NPZ key names.
# The displayed columns show exactly what Notebook 19 stored for each unitig.

with np.load(
    NB19_ASSIGNMENT,
    allow_pickle=True,
) as archive:

    mapping_arrays = {}

    print("Notebook 19 NPZ keys:")

    for key in archive.files:
        arr = np.asarray(
            archive[
                key
            ]
        )

        print(
            f"  {key}: shape={arr.shape}, dtype={arr.dtype}"
        )

        if (
            arr.ndim == 1
            and len(arr) == EXPECTED_UNITIGS
        ):
            mapping_arrays[
                key
            ] = arr

assert len(
    mapping_arrays
) > 0, (
    "Notebook 19 assignment file contained no one-dimensional "
    "unitig-length arrays."
)

annotation_rows = []

significant_lookup = (
    significant
    .set_index(
        "pattern_id"
    )
)

for pattern_id in SIGNIFICANT_PATTERN_IDS:
    pattern_id = int(
        pattern_id
    )

    association_row = significant_lookup.loc[
        pattern_id
    ]

    for unitig_index in pattern_unitig_indices[
        pattern_id
    ]:
        unitig_index = int(
            unitig_index
        )

        record = sequence_by_index[
            unitig_index
        ]

        row = {
            "pattern_id": pattern_id,
            "unitig_index": unitig_index,
            "unitig_header": record[
                "header"
            ],
            "unitig_sequence": record[
                "sequence"
            ],
            "unitig_length": len(
                record[
                    "sequence"
                ]
            ),
            "pattern_n_unitigs": int(
                association_row[
                    "n_unitigs"
                ]
            ),
            "present_count": int(
                association_row[
                    "present_count"
                ]
            ),
            "high_MIC_count": int(
                association_row[
                    "high_MIC_count"
                ]
            ),
            "remaining_count": int(
                association_row[
                    "remaining_count"
                ]
            ),
            "estimated_log2_MIC_difference": float(
                association_row[
                    "estimated_log2_MIC_difference"
                ]
            ),
            "p_value_two_sided": float(
                association_row[
                    "p_value_two_sided"
                ]
            ),
            "Benjamini_Hochberg_q_value": float(
                association_row[
                    "Benjamini_Hochberg_q_value"
                ]
            ),
            "estimated_direction": str(
                association_row[
                    "estimated_direction"
                ]
            ),
        }

        for key, arr in mapping_arrays.items():
            value = arr[
                unitig_index
            ]

            if isinstance(
                value,
                np.generic,
            ):
                value = value.item()

            row[
                f"NB19_{key}"
            ] = value

        annotation_rows.append(
            row
        )

annotation = pd.DataFrame(
    annotation_rows
)

annotation.to_csv(
    OUTPUT_TABLE,
    index=False,
)

print(
    "\nSignificant-pattern unitig annotation:"
)

display(
    annotation
)

print(
    "\nSaved:",
    OUTPUT_TABLE,
)

print(
    "\nTransition: Cell F3A.6 will perform final QC and freeze "
    "the exact pattern/unitig annotation for biological interpretation."
)


In [ ]:
#@title Cell F3A.6 - Final QC and freeze significant-pattern annotation
# Purpose:
# Confirm that the three BH-significant patterns and all their unitigs
# have been recovered and annotated with the existing Notebook 19
# reference-mapping fields.
#
# Nearby/overlapping gene interpretation should be added only after the
# exact mapping fields above are reviewed.

assert set(
    annotation[
        "pattern_id"
    ].astype(
        int
    )
) == set(
    SIGNIFICANT_PATTERN_IDS.tolist()
)

expected_unitig_rows = int(
    significant[
        "n_unitigs"
    ].sum()
)

assert len(
    annotation
) == expected_unitig_rows

assert annotation[
    "unitig_sequence"
].notna().all()

assert OUTPUT_TABLE.exists()

qc = pd.DataFrame(
    [
        {
            "BH_significant_patterns": 3,
            "pattern_ids": "7319;498705;2846",
            "significant_pattern_unitig_rows": len(
                annotation
            ),
            "Notebook17_statistics_verified": True,
            "unitig_to_pattern_mapping_verified": True,
            "unitig_sequences_recovered": True,
            "Notebook19_mapping_fields_attached": True,
            "new_association_test_performed": False,
            "final_QC_pass": True,
        }
    ]
)

qc.to_csv(
    QC_FILE,
    index=False,
)

print("Final QC: PASS")

display(
    qc
)

print(
    "\nAnnotation notebook complete."
)

print(
    "Next step: review the Notebook 19 mapping fields for these unitigs. "
    "Then annotate the exact overlapping/nearby genes without guessing."
)
